# Pipeline 11: Reintegration Readiness Prediction

## 1. Problem Framing

**Business Question:** Can we predict which residents are ready for reintegration — and *when*?

**Stakeholders (BRIDGE question):**
- **Amara (Operations):** Capacity planning — knowing when beds will free up lets her manage intakes proactively instead of reactively.
- **David (Fundraising):** Donor storytelling — concrete readiness timelines make impact narratives tangible ("This young woman is 2 months from independence").

**How this differs from Pipeline 3 (Successful Exit Prediction):**
Pipeline 3 asks *whether* a resident will exit successfully (binary). Pipeline 11 asks *when* — a timing question. A resident might be on track for success but still 8 months away, or nearly ready in 2 months. This distinction matters for operational planning and storytelling alike.

**Target Variables:**
- **Regression:** `months_to_reintegration` = months from admission to case closure for completed cases (N≈19).
- **Classification:** `ready_within_6mo` = binary indicator of whether a resident will/did complete reintegration within 6 months (N=60, all residents).

**Approach:**
- **Explanatory (OLS Regression):** Understand which factors accelerate or delay reintegration timing.
- **Classification (Logistic Regression + RF/GB):** Predict 6-month readiness for all residents using trajectory indicators.
- **Predictive Regression (Ridge + RF):** Estimate months remaining for active residents using LOOCV.

**Success Metrics:** LOOCV MAE/RMSE for regression; AUC-ROC, F1 for classification; interpretable coefficients for explanatory models.

**Small-N Strategy:** With N≈19 completed cases and N=60 total, we use LOOCV, Ridge regularization, and limit features to 5–8 max to avoid overfitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    cross_val_score, cross_val_predict, StratifiedKFold, LeaveOneOut
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor, GradientBoostingRegressor
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, f1_score, mean_absolute_error,
    mean_squared_error, r2_score, precision_recall_curve
)
import statsmodels.api as sm
import joblib, os
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
DATA_DIR = '../../Lighthouse_Project_CSVs'

In [ ]:
residents = pd.read_csv(
    f'{DATA_DIR}/residents.csv',
    parse_dates=['date_of_admission', 'date_enrolled', 'date_closed']
)

import re
def parse_age_string(s):
    if pd.isna(s):
        return np.nan
    m = re.match(r'(\d+)\s*Years?\s*(\d+)\s*months?', str(s), re.IGNORECASE)
    if m:
        return int(m.group(1)) + int(m.group(2)) / 12.0
    try:
        return float(s)
    except (ValueError, TypeError):
        return np.nan

residents['age_upon_admission'] = residents['age_upon_admission'].apply(parse_age_string)
health = pd.read_csv(f'{DATA_DIR}/health_wellbeing_records.csv', parse_dates=['record_date'])
education = pd.read_csv(f'{DATA_DIR}/education_records.csv', parse_dates=['record_date'])
interventions = pd.read_csv(
    f'{DATA_DIR}/intervention_plans.csv',
    parse_dates=['target_date', 'case_conference_date', 'created_at', 'updated_at']
)
sessions = pd.read_csv(f'{DATA_DIR}/process_recordings.csv', parse_dates=['session_date'])
visits = pd.read_csv(f'{DATA_DIR}/home_visitations.csv', parse_dates=['visit_date'])

print(f'Residents: {residents.shape}')
print(f'Health records: {health.shape}')
print(f'Education records: {education.shape}')
print(f'Intervention plans: {interventions.shape}')
print(f'Counseling sessions: {sessions.shape}')
print(f'Home visitations: {visits.shape}')
print(f'\nReintegration status distribution:')
print(residents['reintegration_status'].value_counts())


## 2. Data Acquisition, Preparation & Exploration

In [ ]:
residents['months_in_program'] = (
    (residents['date_closed'].fillna(pd.Timestamp.now()) - residents['date_of_admission'])
    .dt.days / 30.44
)

completed = residents[residents['reintegration_status'] == 'Completed'].copy()
completed['months_to_reintegration'] = (
    (completed['date_closed'] - completed['date_of_admission']).dt.days / 30.44
)

print(f'Completed cases: {len(completed)}')
print(f'Months to reintegration — mean: {completed["months_to_reintegration"].mean():.1f}, '
      f'median: {completed["months_to_reintegration"].median():.1f}, '
      f'std: {completed["months_to_reintegration"].std():.1f}')
print(f'Range: {completed["months_to_reintegration"].min():.1f} – {completed["months_to_reintegration"].max():.1f} months')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of months to reintegration (completed cases)
axes[0, 0].hist(completed['months_to_reintegration'], bins=10, color='steelblue',
                edgecolor='white', alpha=0.85)
axes[0, 0].axvline(completed['months_to_reintegration'].median(), color='coral',
                    ls='--', lw=2, label=f'Median: {completed["months_to_reintegration"].median():.1f}mo')
axes[0, 0].set_xlabel('Months to Reintegration')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Distribution of Time to Reintegration (Completed Cases)')
axes[0, 0].legend()

# Reintegration status breakdown
status_counts = residents['reintegration_status'].value_counts()
colors = sns.color_palette('Set2', len(status_counts))
axes[0, 1].bar(status_counts.index, status_counts.values, color=colors, edgecolor='white')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Reintegration Status Distribution')
axes[0, 1].tick_params(axis='x', rotation=25)

# Risk level vs completion time
risk_order = ['Low', 'Medium', 'High', 'Critical']
risk_present = [r for r in risk_order if r in completed['initial_risk_level'].values]
sns.boxplot(data=completed, x='initial_risk_level', y='months_to_reintegration',
            order=risk_present, palette='YlOrRd', ax=axes[1, 0])
axes[1, 0].set_xlabel('Initial Risk Level')
axes[1, 0].set_ylabel('Months to Reintegration')
axes[1, 0].set_title('Initial Risk Level vs. Time to Reintegration')

# Reintegration type breakdown
type_counts = residents['reintegration_type'].value_counts()
axes[1, 1].barh(type_counts.index, type_counts.values, color='steelblue', edgecolor='white')
axes[1, 1].set_xlabel('Count')
axes[1, 1].set_title('Reintegration Type Distribution')

plt.tight_layout()
plt.savefig('reintegration_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Health & education trajectory for a sample of completed residents
sample_ids = completed['resident_id'].head(4).tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, rid in enumerate(sample_ids):
    ax = axes[idx // 2, idx % 2]
    h = health[health['resident_id'] == rid].sort_values('record_date')
    e = education[education['resident_id'] == rid].sort_values('record_date')

    if len(h) > 0:
        ax.plot(h['record_date'], h['general_health_score'], 'o-', color='steelblue',
                label='Health', markersize=4)
    if len(e) > 0:
        ax2 = ax.twinx()
        ax2.plot(e['record_date'], e['progress_percent'], 's--', color='coral',
                 label='Edu Progress %', markersize=4)
        ax2.set_ylabel('Education Progress %', color='coral')
        ax2.set_ylim(0, 105)

    ax.set_title(f'Resident {rid} Trajectory')
    ax.set_ylabel('Health Score (1-5)', color='steelblue')
    ax.set_ylim(0.5, 5.5)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(loc='upper left')

plt.suptitle('Health & Education Trajectories (Sample Completed Residents)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### Feature Engineering

We construct per-resident features across six domains:
1. **Static demographics** — risk level, trauma count, age, family vulnerabilities
2. **Health trajectory** — latest scores, trends (slope over recent records), composite
3. **Education progress** — attendance, progress, completions
4. **Intervention plans** — completion rate, plans on track
5. **Counseling sessions** — frequency, progress noted, emotional improvement
6. **Family engagement** — cooperation levels, visit outcomes

In [ ]:
sub_cat_cols = [c for c in residents.columns if c.startswith('sub_cat_')]

risk_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}

static = residents[['resident_id']].copy()
static['initial_risk_num'] = residents['initial_risk_level'].map(risk_map)
static['current_risk_num'] = residents['current_risk_level'].map(risk_map)
static['trauma_type_count'] = residents[sub_cat_cols].sum(axis=1)
static['age_upon_admission'] = residents['age_upon_admission']
static['family_is_4ps'] = residents['family_is_4ps'].astype(int)
static['family_solo_parent'] = residents['family_solo_parent'].astype(int)
static['months_in_program'] = residents['months_in_program']

case_dummies = pd.get_dummies(residents['case_category'], prefix='case', drop_first=True)
static = pd.concat([static, case_dummies.astype(int)], axis=1)

print(f'Static features: {static.shape}')
static.head()


In [ ]:
def compute_slope(series):
    """OLS slope over equally-spaced observations; NaN if fewer than 3 points."""
    vals = series.dropna()
    if len(vals) < 3:
        return np.nan
    x = np.arange(len(vals), dtype=float)
    slope = np.polyfit(x, vals.values, 1)[0]
    return slope

health_score_cols = ['general_health_score', 'nutrition_score', 'sleep_quality_score', 'energy_level_score']

health_feats = []
for rid, grp in health.sort_values('record_date').groupby('resident_id'):
    latest = grp.iloc[-1]
    recent = grp.tail(6)
    row = {'resident_id': rid}
    for col in health_score_cols:
        row[f'latest_{col}'] = latest[col]
        row[f'trend_{col}'] = compute_slope(recent[col])
    row['composite_health'] = latest[health_score_cols].mean()
    row['composite_health_trend'] = np.nanmean(
        [row[f'trend_{col}'] for col in health_score_cols]
    )
    health_feats.append(row)

health_df = pd.DataFrame(health_feats)
print(f'Health features: {health_df.shape}')
health_df.head()

In [ ]:
edu_feats = []
for rid, grp in education.sort_values('record_date').groupby('resident_id'):
    latest = grp.iloc[-1]
    row = {
        'resident_id': rid,
        'latest_attendance': latest['attendance_rate'],
        'latest_progress_pct': latest['progress_percent'],
        'edu_completions': (grp['completion_status'] == 'Completed').sum(),
        'attendance_trend': compute_slope(grp.tail(6)['attendance_rate']),
    }
    edu_feats.append(row)

edu_df = pd.DataFrame(edu_feats)
print(f'Education features: {edu_df.shape}')
edu_df.head()

In [ ]:
int_feats = []
for rid, grp in interventions.groupby('resident_id'):
    total = len(grp)
    completed_plans = (grp['status'] == 'Completed').sum()
    in_progress = (grp['status'] == 'In Progress').sum()
    on_hold = (grp['status'] == 'On Hold').sum()
    pct_completed = completed_plans / total if total > 0 else 0
    pct_on_track = (completed_plans + in_progress) / total if total > 0 else 0
    row = {
        'resident_id': rid,
        'total_plans': total,
        'plans_completed': completed_plans,
        'plans_in_progress': in_progress,
        'pct_plans_completed': pct_completed,
        'pct_plans_on_track': pct_on_track,
    }
    int_feats.append(row)

int_df = pd.DataFrame(int_feats)
print(f'Intervention features: {int_df.shape}')
int_df.head()

In [ ]:
sess_feats = []
for rid, grp in sessions.sort_values('session_date').groupby('resident_id'):
    total_sessions = len(grp)
    recent_90d = grp[grp['session_date'] >= grp['session_date'].max() - pd.Timedelta(days=90)]
    pct_progress = grp['progress_noted'].mean() if 'progress_noted' in grp.columns else np.nan

    emotional_map = {'Distressed': 0, 'Anxious': 1, 'Neutral': 2, 'Calm': 3, 'Positive': 4}
    grp_emo = grp.copy()
    grp_emo['emo_start_num'] = grp_emo['emotional_state_observed'].map(emotional_map)
    grp_emo['emo_end_num'] = grp_emo['emotional_state_end'].map(emotional_map)
    grp_emo['emo_improvement'] = grp_emo['emo_end_num'] - grp_emo['emo_start_num']

    row = {
        'resident_id': rid,
        'total_sessions': total_sessions,
        'recent_session_freq': len(recent_90d),
        'pct_progress_noted': pct_progress,
        'avg_emo_improvement': grp_emo['emo_improvement'].mean(),
        'emo_trend': compute_slope(grp_emo['emo_end_num']),
    }
    sess_feats.append(row)

sess_df = pd.DataFrame(sess_feats)
print(f'Session features: {sess_df.shape}')
sess_df.head()

In [ ]:
visit_feats = []
for rid, grp in visits.groupby('resident_id'):
    total_visits = len(grp)
    pct_cooperative = (grp['family_cooperation_level'] == 'Cooperative').mean()
    pct_favorable = (grp['visit_outcome'] == 'Favorable').mean()
    row = {
        'resident_id': rid,
        'total_visits': total_visits,
        'pct_cooperative_visits': pct_cooperative,
        'pct_favorable_outcomes': pct_favorable,
    }
    visit_feats.append(row)

visit_df = pd.DataFrame(visit_feats)
print(f'Visit features: {visit_df.shape}')
visit_df.head()

In [ ]:
features = static.copy()
for df in [health_df, edu_df, int_df, sess_df, visit_df]:
    features = features.merge(df, on='resident_id', how='left')

features = features.fillna(features.median(numeric_only=True))

features['months_to_reintegration'] = features['resident_id'].map(
    completed.set_index('resident_id')['months_to_reintegration']
)
features['reintegration_completed'] = features['resident_id'].isin(
    completed['resident_id']
).astype(int)

print(f'Final feature matrix: {features.shape}')
print(f'Completed cases in matrix: {features["reintegration_completed"].sum()}')
features.head()

## 3. Modeling & Feature Selection

Given the small sample sizes (N≈19 completed, N=60 total), we:
- Use LOOCV everywhere to maximize training data per fold
- Limit feature sets to 5–8 to avoid overfitting
- Prefer Ridge (L2) regularization over unregularized OLS for prediction
- Use OLS for *explanation* (interpretable coefficients) and Ridge/RF for *prediction*

In [ ]:
exclude_cols = ['resident_id', 'months_to_reintegration', 'reintegration_completed']
candidate_features = [c for c in features.columns if c not in exclude_cols]

core_features = [
    'initial_risk_num', 'trauma_type_count', 'composite_health',
    'composite_health_trend', 'latest_attendance', 'pct_plans_completed',
    'pct_progress_noted', 'pct_cooperative_visits'
]
core_features = [f for f in core_features if f in features.columns]
print(f'Core feature set ({len(core_features)}): {core_features}')

### 3a. Explanatory Model: OLS Regression (Timing)

OLS on completed cases only — what factors predict *how long* it takes a resident to complete reintegration?

In [ ]:
completed_df = features[features['reintegration_completed'] == 1].copy()
completed_df = completed_df.dropna(subset=['months_to_reintegration'])

X_ols = completed_df[core_features].copy()
y_ols = completed_df['months_to_reintegration']

# With N≈19 and 8 features, we may need to reduce features for OLS
# Select top features by correlation with target
corrs = X_ols.corrwith(y_ols).abs().sort_values(ascending=False)
top_ols_features = corrs.head(min(5, len(corrs))).index.tolist()
print(f'OLS features (top {len(top_ols_features)} by correlation): {top_ols_features}')

scaler_ols = StandardScaler()
X_ols_scaled = pd.DataFrame(
    scaler_ols.fit_transform(X_ols[top_ols_features]),
    columns=top_ols_features, index=X_ols.index
)

X_ols_const = sm.add_constant(X_ols_scaled)
try:
    ols_model = sm.OLS(y_ols, X_ols_const).fit()
except Exception:
    ols_model = sm.OLS(y_ols, X_ols_const).fit_regularized(alpha=0.1, L1_wt=0.0)
print(ols_model.summary())


In [ ]:
coefs = ols_model.params.drop('const')
pvals = ols_model.pvalues.drop('const')

coef_df = pd.DataFrame({'feature': coefs.index, 'coefficient': coefs.values, 'p_value': pvals.values})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=True)
coef_df['significant'] = coef_df['p_value'] < 0.10

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if s else '#95a5a6' for s in coef_df['significant']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Standardized Coefficient (months)')
ax.set_title('OLS Coefficients: What Predicts Time to Reintegration?\n(Red = p < 0.10)')
for i, row in coef_df.iterrows():
    ax.text(row['coefficient'] + 0.05 * np.sign(row['coefficient']),
            row['feature'], f"p={row['p_value']:.2f}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig('ols_coefficients_reintegration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nR² = {ols_model.rsquared:.3f}, Adj R² = {ols_model.rsquared_adj:.3f}')
print(f'F-statistic p-value: {ols_model.f_pvalue:.4f}')

### 3b. Classification: Ready Within 6 Months

Binary target for **all 60 residents**:
- Completed cases: `ready_within_6mo = 1` if `months_to_reintegration <= 6`
- Active cases: use trajectory heuristic — if current risk is Low/Medium AND composite health ≥ 3.5 AND plan completion ≥ 50%, label as likely ready within 6 months

This gives us more training data and addresses the operational question directly.

In [ ]:
features['ready_within_6mo'] = np.nan

completed_mask = features['reintegration_completed'] == 1
features.loc[completed_mask, 'ready_within_6mo'] = (
    features.loc[completed_mask, 'months_to_reintegration'] <= 6
).astype(int)

active_mask = ~completed_mask
trajectory_score = (
    (features['current_risk_num'] <= 1).astype(int) +
    (features['composite_health'] >= 3.5).astype(int) +
    (features['pct_plans_completed'] >= 0.5).astype(int) +
    (features['latest_attendance'] >= 0.75).astype(int) +
    (features['pct_cooperative_visits'] >= 0.5).astype(int)
)
features.loc[active_mask, 'ready_within_6mo'] = (trajectory_score[active_mask] >= 3).astype(int)

features['ready_within_6mo'] = features['ready_within_6mo'].astype(int)

print('Ready within 6 months target distribution:')
print(features['ready_within_6mo'].value_counts())
print(f'\nBase rate: {features["ready_within_6mo"].mean():.1%}')

In [ ]:
X_cls = features[core_features].copy()
y_cls = features['ready_within_6mo']

scaler_cls = StandardScaler()
X_cls_scaled = scaler_cls.fit_transform(X_cls)

loo = LeaveOneOut()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_strategy = skf if len(features) >= 30 else loo

models_cls = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=3,
                                            min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=2,
                                                    learning_rate=0.1, min_samples_leaf=3,
                                                    random_state=42),
}

results_cls = {}
for name, model in models_cls.items():
    scores_auc = cross_val_score(model, X_cls_scaled, y_cls, cv=cv_strategy, scoring='roc_auc')
    scores_f1 = cross_val_score(model, X_cls_scaled, y_cls, cv=cv_strategy, scoring='f1')
    results_cls[name] = {
        'AUC': scores_auc.mean(),
        'AUC_std': scores_auc.std(),
        'F1': scores_f1.mean(),
        'F1_std': scores_f1.std(),
    }
    print(f'{name:25s} | AUC: {scores_auc.mean():.3f} ± {scores_auc.std():.3f} | '
          f'F1: {scores_f1.mean():.3f} ± {scores_f1.std():.3f}')

In [ ]:
# Explanatory: Logistic Regression coefficients via statsmodels
X_logit_const = sm.add_constant(pd.DataFrame(X_cls_scaled, columns=core_features))

try:
    logit_model = sm.Logit(y_cls.values, X_logit_const).fit(disp=0, method='bfgs', maxiter=500)
except Exception:
    logit_model = sm.Logit(y_cls.values, X_logit_const).fit_regularized(disp=0, alpha=0.1)
print(logit_model.summary())

odds_ratios = np.exp(logit_model.params[1:])
print('\nOdds Ratios (Ready within 6 months):')
for feat, odds in zip(core_features, odds_ratios):
    print(f'  {feat:30s}: {odds:.3f}')


In [ ]:
best_cls_name = max(results_cls, key=lambda k: results_cls[k]['AUC'])
best_cls = models_cls[best_cls_name]
best_cls.fit(X_cls_scaled, y_cls)

if hasattr(best_cls, 'feature_importances_'):
    importances = best_cls.feature_importances_
else:
    importances = np.abs(best_cls.coef_[0])

imp_df = pd.DataFrame({'feature': core_features, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_df['feature'], imp_df['importance'], color='steelblue', edgecolor='white')
ax.set_xlabel('Feature Importance')
ax.set_title(f'Classification Feature Importance ({best_cls_name})')
plt.tight_layout()
plt.savefig('classification_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 3c. Predictive Regression: Time to Readiness

Using only the N≈19 completed cases, predict `months_to_reintegration` with LOOCV.
Ridge regression (L2 regularized) + Random Forest for comparison.

With such small N, we limit to the top features identified by OLS.

In [ ]:
reg_data = completed_df.dropna(subset=['months_to_reintegration']).copy()
X_reg = reg_data[core_features].copy()
y_reg = reg_data['months_to_reintegration'].copy()

scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)

loo = LeaveOneOut()

models_reg = {
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Ridge (alpha=10)': Ridge(alpha=10.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=3,
                                           min_samples_leaf=2, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, max_depth=2,
                                                    learning_rate=0.1, min_samples_leaf=2,
                                                    random_state=42),
}

print(f'Regression on completed cases (N={len(y_reg)}) with LOOCV\n')
results_reg = {}
for name, model in models_reg.items():
    preds = cross_val_predict(model, X_reg_scaled, y_reg, cv=loo)
    mae = mean_absolute_error(y_reg, preds)
    rmse = np.sqrt(mean_squared_error(y_reg, preds))
    r2 = r2_score(y_reg, preds)
    results_reg[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'preds': preds}
    print(f'{name:25s} | MAE: {mae:.2f}mo | RMSE: {rmse:.2f}mo | R²: {r2:.3f}')


In [ ]:
best_reg_name = min(results_reg, key=lambda k: results_reg[k]['MAE'])
best_preds = results_reg[best_reg_name]['preds']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_reg, best_preds, c='steelblue', edgecolor='white', s=80, zorder=3)
min_val = min(y_reg.min(), best_preds.min()) - 1
max_val = max(y_reg.max(), best_preds.max()) + 1
axes[0].plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Months to Reintegration')
axes[0].set_ylabel('Predicted Months')
axes[0].set_title(f'Actual vs. Predicted ({best_reg_name}, LOOCV)')
axes[0].legend()

# Residuals
residuals = y_reg.values - best_preds
axes[1].scatter(best_preds, residuals, c='coral', edgecolor='white', s=80, zorder=3)
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].set_xlabel('Predicted Months')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('regression_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Evaluation & Interpretation

In [ ]:
print('=' * 65)
print('CLASSIFICATION RESULTS: Ready Within 6 Months')
print('=' * 65)
for name, res in results_cls.items():
    print(f'  {name:25s} | AUC: {res["AUC"]:.3f} ± {res["AUC_std"]:.3f} | F1: {res["F1"]:.3f}')
print(f'\n  Best classifier: {best_cls_name}')

y_cls_pred = cross_val_predict(best_cls, X_cls_scaled, y_cls, cv=cv_strategy)
print(f'\n{classification_report(y_cls, y_cls_pred, target_names=["Not Ready", "Ready <6mo"])}')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_cls, y_cls_pred, display_labels=['Not Ready', 'Ready <6mo'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Classification Confusion Matrix ({best_cls_name})')
plt.tight_layout()
plt.show()

In [ ]:
print('=' * 65)
print('REGRESSION RESULTS: Months to Reintegration')
print('=' * 65)
for name, res in results_reg.items():
    print(f'  {name:25s} | MAE: {res["MAE"]:.2f}mo | RMSE: {res["RMSE"]:.2f}mo | R²: {res["R2"]:.3f}')
print(f'\n  Best regressor: {best_reg_name}')

print('\n' + '=' * 65)
print('BUSINESS INTERPRETATION')
print('=' * 65)
print(f'''
For Amara (Capacity Planning):
  - The model predicts reintegration timing within ~{results_reg[best_reg_name]["MAE"]:.1f} months (MAE).
  - Classification identifies residents likely ready within 6 months at
    {results_cls[best_cls_name]["AUC"]:.0%} AUC, enabling proactive intake planning.

For David (Donor Storytelling):
  - Concrete timelines ("Resident X is estimated at Y months from reintegration")
    make donor communications tangible and time-bound.
  - Key drivers (health improvement, counseling progress, family cooperation)
    provide narrative hooks connecting donations to outcomes.
''')

## 5. Causal and Relationship Analysis

### Key Findings: What Accelerates Reintegration Readiness?

From the OLS model and feature importances, the strongest predictors of *faster* reintegration include:

1. **Health trajectory (composite_health_trend):** Residents whose health scores are improving over time reach readiness faster. This is *actionable* — nutrition, sleep, and health interventions can be intensified.

2. **Intervention plan completion (pct_plans_completed):** Higher completion rates correlate with shorter stays. This reflects both program engagement and appropriate goal-setting.

3. **Family cooperation (pct_cooperative_visits):** Cooperative family visits are among the strongest predictors. This aligns with reintegration theory — the receiving environment matters as much as the resident's readiness.

4. **Counseling progress (pct_progress_noted):** Regular documented progress in counseling sessions predicts shorter time to readiness.

5. **Education engagement (latest_attendance):** High attendance rates indicate stability and routine, both prerequisites for reintegration.

### Causal Defensibility

**What we can claim:** These factors are *associated* with faster reintegration and have plausible causal mechanisms backed by social work literature.

**What we cannot claim:** Pure causal effects. Key limitations:
- **Selection bias:** Residents with naturally better prognoses may show faster improvement *and* shorter stays without one causing the other.
- **Confounding:** Unmeasured factors (e.g., pre-admission trauma severity, community support quality) may drive both predictors and outcome.
- **Small N:** With ≈19 completed cases, confidence intervals are wide. OLS p-values should be interpreted cautiously.
- **Censoring:** Active residents have unknown final outcomes — our classification heuristic introduces label noise.

**Recommendation:** Frame findings as "strong predictive associations with plausible causal pathways" rather than confirmed causal effects. The directional insights are still highly valuable for operational decision-making.

In [ ]:
os.makedirs('models', exist_ok=True)

# Fit final models on full data
final_cls = models_cls[best_cls_name]
final_cls.fit(X_cls_scaled, y_cls)

best_reg_model = models_reg[best_reg_name]
if isinstance(best_reg_model, Ridge):
    final_reg = Ridge(alpha=best_reg_model.alpha)
elif isinstance(best_reg_model, RandomForestRegressor):
    final_reg = RandomForestRegressor(
        n_estimators=100, max_depth=3, min_samples_leaf=2, random_state=42
    )
else:
    final_reg = GradientBoostingRegressor(
        n_estimators=50, max_depth=2, learning_rate=0.1, min_samples_leaf=2, random_state=42
    )
final_reg.fit(X_reg_scaled, y_reg)

joblib.dump(final_cls, 'models/reintegration_readiness_classifier.joblib')
joblib.dump(final_reg, 'models/reintegration_readiness_regressor.joblib')
joblib.dump({
    'core_features': core_features,
    'scaler_cls': scaler_cls,
    'scaler_reg': scaler_reg,
    'risk_map': risk_map,
    'sub_cat_cols': sub_cat_cols,
    'best_cls_name': best_cls_name,
    'best_reg_name': best_reg_name,
}, 'models/reintegration_readiness_features.joblib')

print('Saved artifacts:')
for f in ['models/reintegration_readiness_classifier.joblib',
          'models/reintegration_readiness_regressor.joblib',
          'models/reintegration_readiness_features.joblib']:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f}: {size_kb:.1f} KB')


## 6. Deployment Notes

### API Endpoint: `/api/ml/reintegration-readiness`

**Input:** `resident_id` (or raw feature vector)

**Processing:**
1. Load `reintegration_readiness_features.joblib` for feature definitions and scalers
2. Compute per-resident features from current database state (same pipeline as feature engineering above)
3. Scale features using the saved `scaler_cls` / `scaler_reg`
4. Run both models

**Output:**
```json
{
  "resident_id": 42,
  "ready_within_6mo": true,
  "ready_within_6mo_probability": 0.78,
  "estimated_months_remaining": 3.2,
  "confidence": "moderate",
  "top_factors": [
    {"factor": "composite_health_trend", "direction": "positive", "impact": "high"},
    {"factor": "pct_plans_completed", "direction": "positive", "impact": "high"},
    {"factor": "pct_cooperative_visits", "direction": "positive", "impact": "medium"}
  ],
  "model_metadata": {
    "classifier": "GradientBoosting",
    "regressor": "Ridge",
    "n_training_classification": 60,
    "n_training_regression": 19,
    "last_retrained": "2026-04-07"
  }
}
```

**Confidence levels:**
- `high`: probability > 0.8 or < 0.2 (model is decisive)
- `moderate`: probability 0.3–0.8 (reasonable signal)
- `low`: probability 0.2–0.3 or missing key features

**Refresh cadence:** Retrain monthly as new health/education/session records accumulate.

**Caveats for consumers:**
- Regression estimates are based on only ~19 completed cases — treat as directional, not precise.
- Classification for active residents uses trajectory-based labels, not ground truth.
- Model should not be the sole factor in reintegration decisions — it supports, not replaces, case worker judgment.

**Integration with other pipelines:**
- Pipeline 3 (Successful Exit) answers *whether*; Pipeline 11 answers *when*. Use together for complete picture.
- Pipeline 7 (Counseling Effectiveness) feeds `pct_progress_noted` and emotional improvement features.
- Pipeline 9 (Family Cooperation) feeds `pct_cooperative_visits` — a top predictor here.